In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

DATA_DIR = os.path.join(r"D:\Upskill\Mini_Projects\intelligent-predictive-maintenance-system\CMAPSS_Data")

In [ ]:
train_data = pd.read_csv(os.path.join(DATA_DIR, "train_FD001.txt"), sep=" ", header=None)
train_data

In [ ]:
train_data = train_data.dropna(axis=1)
train_data

In [ ]:
#Get the column names from the dataset documentation
column_names = ["engine_id", "cycle"] + [f"operational_setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
column_names

In [ ]:
#Assign column names to the DataFrame
train_data.columns = column_names
train_data

In [ ]:
# Plotting the sensor data for 1 engine
# Some sensor values constant -> Useless
engine_1 = train_data[train_data['engine_id'] == 1]
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]

engine_1.plot(
    x = "cycle",
    y = sensor_cols,
    legend = True
)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

In [ ]:
# Finding RUL (Remaining Useful Life) using the given data
max_cycles = train_data.groupby('engine_id')["cycle"].max() #Give max cycles of each engine as a pandas Series
print(max_cycles)
print('---------------------------------------------')

# Adding an RUL column that tells how close an engine is to failure. Creating ground truth for the training
train_data['RUL'] = train_data.apply(lambda row:max_cycles[row['engine_id']] - row['cycle'], axis = 1)
print(train_data[['engine_id','cycle', 'RUL']])
print('---------------------------------------------')

# Adding a boolean column to tell if the engine is close to failure or not based on a threshold
threshold = 30
train_data['failure_risk'] = (train_data['RUL'] <= 30).astype(int) # Convert to int as we need numeric data to train model
print(train_data.loc[train_data['engine_id'] == 1, ['engine_id','cycle', 'RUL', 'failure_risk']]) #.loc[] takes input as [row, column]. filter by row first then by column

In [ ]:
# Adding RUL for test dataset in the same way

test_data = pd.read_csv(os.path.join(DATA_DIR, "test_FD001.txt"), sep=" ", header=None)
RUL_data = pd.read_csv(os.path.join(DATA_DIR, "RUL_FD001.txt"), header=None)

test_data.dropna(axis=1, inplace=True)
RUL_data.dropna(axis=1, inplace=True)

test_data.columns = column_names

max_cycles = test_data.groupby('engine_id')['cycle'].max()
max_cycles = max_cycles + RUL_data[0].values

test_data['RUL'] = test_data.apply(lambda row:max_cycles[row['engine_id']] - row['cycle'], axis = 1)

threshold = 30
test_data['failure_risk'] = (test_data['RUL'] <= 30).astype(int) # Convert to int as we need numeric data to train model

In [ ]:
print(train_data.shape)
print(test_data.shape)

In [ ]:
print(train_data['engine_id'].nunique())
print(test_data['engine_id'].nunique())

### Starting with PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, lag, stddev, col
import os, sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

In [ ]:
#Creating spark session
spark = (
    SparkSession.builder
    .master("local[1]")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "true")
    .config("spark.executor.instances", "1")
    .getOrCreate()
)

In [ ]:
#Converting pandas df to Spark df

train_spark = spark.createDataFrame(train_data)
test_spark = spark.createDataFrame(test_data)

In [ ]:
train_spark.show()

In [ ]:
# Define a window over which all operations will be performed
window = Window.partitionBy("engine_id").orderBy("cycle") # For each engine, compute features in time order

# Define a rolling window
rolling_window = Window.partitionBy("engine_id") \
                       .orderBy("cycle") \
                       .rowsBetween(-4, 0) #current row + previous 4 rows

In [ ]:
#Computing lag1, lag2, mean, and std dev for each sensor in training dataframe

for sensor in sensor_cols:

    train_spark = train_spark.withColumn(
        sensor,
        col(sensor).cast("double")
    )

    # Lag 1 (cycle 10 → value from cycle 9)
    train_spark = train_spark.withColumn(
        f"{sensor}_lag1",
        lag(sensor, 1).over(window)
    )

    # Lag 2 (cycle 10 → value from cycle 9)
    train_spark = train_spark.withColumn(
        f"{sensor}_lag2",
        lag(sensor, 2).over(window)
    )

    # Rolling Mean
    train_spark = train_spark.withColumn(
        f"{sensor}_mean5",
        avg(sensor).over(rolling_window)
    )

    # Rolling Std Dev
    train_spark = train_spark.withColumn(
        f"{sensor}_std5",
        stddev(sensor).over(rolling_window)
    )

train_spark = train_spark.dropna()

In [ ]:
train_spark.filter("engine_id = 1").show(10)

In [ ]:
#Computing lag1, lag2, mean, and std dev for each sensor in testing dataframe

for sensor in sensor_cols:

    test_spark = test_spark.withColumn(
        sensor,
        col(sensor).cast("double")
    )

    # Lag 1 (cycle 10 → value from cycle 9)
    test_spark = test_spark.withColumn(
        f"{sensor}_lag1",
        lag(sensor, 1).over(window)
    )

    # Lag 2 (cycle 10 → value from cycle 9)
    test_spark = test_spark.withColumn(
        f"{sensor}_lag2",
        lag(sensor, 2).over(window)
    )

    # Rolling Mean
    test_spark = test_spark.withColumn(
        f"{sensor}_mean5",
        avg(sensor).over(rolling_window)
    )

    # Rolling Std Dev
    test_spark = test_spark.withColumn(
        f"{sensor}_std5",
        stddev(sensor).over(rolling_window)
    )

test_spark = test_spark.dropna()

In [ ]:
test_spark.filter("engine_id = 1").show(10)

In [ ]:
print("Training shape: ", train_spark.count(), len(train_spark.columns))
print("Testing shape: ", test_spark.count(), len(test_spark.columns))

In [ ]:
# Converting PySpark dataframe back to pandas dataframe to train model
train_df = train_spark.toPandas()
test_df = test_spark.toPandas()

# Model training using Sklearn

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# Separating input from output
y_train = train_df['failure_risk']
y_test = test_df['failure_risk']

drop_columns = ['engine_id', 'cycle', 'RUL', 'failure_risk']
x_train = train_df.drop(columns=drop_columns)
x_test = test_df.drop(columns=drop_columns)

In [ ]:
print("x_train: ", x_train.shape)
print("y_train: ", y_train.shape)
print("x_test: ", x_test.shape)
print("y_test: ", y_test.shape)

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X=x_train_scaled, y=y_train)

y_pred = model.predict(x_test_scaled)
y_prob = model.predict_proba(x_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Class 0: Healthy engine     
Class 1: Engine failure

Class 1 Recall is 0.63 -> 37% of failures are missed

From the confusion matrix, 123 False Negatives, i.e, 123 failures missed.

Class 0 has 12,564 entries and Class 1 has only 332
Highly imbalanced

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X=x_train_scaled, y=y_train)

y_pred = model.predict(x_test_scaled)
y_prob = model.predict_proba(x_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Class 0: Healthy engine     
Class 1: Engine failure

Class 1 Recall is 0.83 -> 17% of failures are missed

From the confusion matrix, 56 False Negatives, i.e, 56 failures missed.

More False Positives than before, i.e, more false alarms about failures.
But false alarm better than missing actual failures.

Setting the class_weight parameter to balanced gives equal weightage to all classes even if there is an imbalance in the number of datapoints.

# Model training using RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)


# Random forest works with unscaled data and it is robust to noisy sensors
rf.fit(x_train, y_train)

y_pred = rf.predict(x_test)
y_prob = rf.predict_proba(x_test)[:,1]

importance = pd.Series(rf.feature_importances_, index=x_train.columns).sort_values(ascending=False) # Tells which sensor predicts failure better


print(classification_report(y_test, y_pred))
print('---------------------------------------------')
print(confusion_matrix(y_test, y_pred))
print('---------------------------------------------')
print(importance.head(20)) 

Class 1 Recall is 0.59 -> 41% of failures are missed. VERY BAD

From the confusion matrix, 135 False Negatives, i.e, 135 failures missed. VERY BAD

Random Forest is more conservative.
it gives better precision than Logistic Regression but worse recall.
i.e, RF gives less false alarms but misses a lot more failures.

For predictive maintenance, the goal is to reduce the missed failures.
So, LR > RF.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)


# Random forest works with unscaled data and it is robust to noisy sensors
rf.fit(x_train, y_train)

y_prob = rf.predict_proba(x_test)[:,1]
y_pred = (y_prob > 0.3).astype(int) #If probability > 0.3, predict as failure. (By default this value is 0.5)

importance = pd.Series(rf.feature_importances_, index=x_train.columns).sort_values(ascending=False) # Tells which sensor predicts failure better


print(classification_report(y_test, y_pred))
print('---------------------------------------------')
print(confusion_matrix(y_test, y_pred))
print('---------------------------------------------')
print(importance.head(20)) 

Class 1 Recall is now 0.74 -> 26% of failures are missed. Better than before

From the confusion matrix, 87 False Negatives, i.e, 87 failures missed.

Setting the threshold lower gives better results because the dataset is unbalanced, having more class 0 elements than class 1.

Model comparison so far:    
Logistic Regression (balanced)          
Recall: 0.83            
Precision: 0.59     
F1: 0.69        

Random Forest (threshold 0.3)   
Recall: 0.74    
Precision: 0.71     
F1: 0.72 ← best overall balance 

Logistic Regression maximized recall for failure detection, while Random Forest provided a better precision-recall balance using probabilistic threshold tuning.